# NYUMets: what contrasts does each patient have?

Discovery-first EDA over `data/imaging/.../<patientId>/`. Nothing here assumes the NYUMets
naming convention -- the early cells **report the filename/directory vocabulary they find**,
and the contrast classifier is a small ordered rule table you edit once you have seen it.

Order of business:

0. locate the root, probe the directory layout
1. walk every patient -> file manifest (cached to CSV)
2. token census: what words actually appear in paths / filenames
3. classify files -> contrast labels; **report what did not match**
4. infer visit / session keys (longitudinal structure)
5. availability tables + plots: patient x contrast, session x contrast, co-occurrence, combos
6. header-only scan: shapes, voxel spacing, orientation, within-session affine agreement
7. per-contrast intensity histograms across subjects
8. eyeball one session
9. write the tidy `(patient, session, contrast)` CSV

Steps 0-5 are stdlib + numpy + matplotlib only (no pandas), so they run in a bare env. From step 6
on it also needs `nibabel` and the repo's own `visualization` helpers (`subplot_hists`,
`subplot_images`), which pull in torch.


## 0. Root + layout probe

In [ ]:
import os, re, sys, csv, json, time
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# repo root (works whether the kernel cwd is ImMAP/ or ImMAP/notebooks/)
REPO = os.getcwd()
if os.path.basename(REPO) == "notebooks":
    REPO = os.path.dirname(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# NYUMets h5s store canonical-RAS axes, so a raw imshow puts the eyes on the image's RIGHT.
# DISPLAY ONLY -- the stored pixels and every checkpoint are untouched.
from visualization.image import set_display_orient
set_display_orient("radiological")           # "neurological", or None to switch it off

# the symlink lives beside the repo; the gpfs path is the fallback
CANDIDATES = [
    os.path.join(REPO, "..", "datasets", "NYUMets"),
    "/gpfs/data/fenglab/LiFeng/NYUMets/NYUMets",
]
ROOT = next((os.path.abspath(p) for p in CANDIDATES if os.path.isdir(p)), None)
assert ROOT is not None, "none of these exist:\n  " + "\n  ".join(map(os.path.abspath, CANDIDATES))

CACHE = os.path.join(REPO, "cache")
os.makedirs(CACHE, exist_ok=True)
MANIFEST_CSV = os.path.join(CACHE, "nyumets_manifest.csv")
TIDY_CSV     = os.path.join(CACHE, "nyumets_contrasts.csv")

# knobs
MAX_PATIENTS = None    # None = all; set e.g. 50 for a fast first pass over a slow FS
COLLECT_SIZE = True    # st_size per file: one extra stat each, but catches empty/truncated volumes

print("ROOT :", ROOT)
print("real :", os.path.realpath(ROOT))
print("cache:", CACHE)

In [ ]:
def tree(root, max_depth=3, max_entries=8, _depth=0, _prefix=""):
    """Depth- and width-limited directory listing. Directories first, files after."""
    try:
        entries = sorted(os.scandir(root), key=lambda e: (not e.is_dir(), e.name))
    except OSError as err:
        print(_prefix + "!! " + str(err))
        return
    shown = entries[:max_entries]
    for e in shown:
        print(_prefix + ("[d] " if e.is_dir() else "    ") + e.name)
        if e.is_dir() and _depth + 1 < max_depth:
            tree(e.path, max_depth, max_entries, _depth + 1, _prefix + "    ")
    if len(entries) > max_entries:
        print(_prefix + f"... (+{len(entries) - max_entries} more)")


print(f"=== {ROOT} ===")
tree(ROOT, max_depth=2, max_entries=12)

### Where do the patient folders start?

The stated layout is `data/imaging/patientId/<ID>/...`. We try that first and fall back through
the plausible parents. Whatever it picks is printed -- **check it before moving on** and hard-set
`PATIENT_ROOT` if the guess is wrong.

In [ ]:
def pick_patient_root(root):
    """First candidate that exists and holds many subdirectories -> the per-patient level."""
    rels = ["data/imaging/patientId", "data/imaging", "data", "imaging", ""]
    best = None
    for rel in rels:
        p = os.path.join(root, *rel.split("/")) if rel else root
        if not os.path.isdir(p):
            continue
        subs = [e.name for e in os.scandir(p) if e.is_dir()]
        print(f"{rel or '.':<24} {len(subs):>6} subdirs   e.g. {sorted(subs)[:4]}")
        if best is None and len(subs) >= 10:
            best = p
    return best


PATIENT_ROOT = pick_patient_root(ROOT)
assert PATIENT_ROOT, "no directory level looked like a patient list -- set PATIENT_ROOT by hand"
print("\nPATIENT_ROOT =", PATIENT_ROOT)

PATIENTS = sorted(e.name for e in os.scandir(PATIENT_ROOT) if e.is_dir())
if MAX_PATIENTS:
    PATIENTS = PATIENTS[:MAX_PATIENTS]
print(f"{len(PATIENTS)} patients   first: {PATIENTS[:5]}   last: {PATIENTS[-3:]}")

In [ ]:
# what does the inside of a patient folder look like?
for pid in PATIENTS[:3]:
    print(f"=== {pid} ===")
    tree(os.path.join(PATIENT_ROOT, pid), max_depth=3, max_entries=10)
    print()

## 1. Walk every patient -> manifest

One row per file, `.nii/.nii.gz` and everything else alike -- the non-image files (JSON sidecars,
CSV metadata, DICOM leftovers) tell you as much about the layout as the volumes do.
Cached to CSV; set `REWALK=True` to redo it.

In [ ]:
REWALK = False

NII_RE = re.compile(r"\.nii(\.gz)?$", re.I)


def ext_of(fname):
    """'.nii.gz' kept whole; otherwise the last suffix. '' for extensionless files."""
    low = fname.lower()
    if low.endswith(".nii.gz"):
        return ".nii.gz"
    return os.path.splitext(low)[1]


def stem_of(fname):
    e = ext_of(fname)
    return fname[: -len(e)] if e else fname


def walk_patients(patient_root, patients, collect_size=True):
    rows = []
    t0 = time.time()
    for i, pid in enumerate(patients):
        pdir = os.path.join(patient_root, pid)
        for dirpath, _dirnames, filenames in os.walk(pdir):
            rel = os.path.relpath(dirpath, pdir)
            rel = "" if rel == "." else rel.replace(os.sep, "/")
            for fn in filenames:
                nbytes = -1
                if collect_size:
                    try:
                        nbytes = os.stat(os.path.join(dirpath, fn)).st_size
                    except OSError:
                        pass
                rows.append({"patient": pid, "reldir": rel, "fname": fn,
                             "ext": ext_of(fn), "nbytes": nbytes})
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(patients)} patients, {len(rows)} files, {time.time()-t0:.0f}s")
    return rows


FIELDS = ["patient", "reldir", "fname", "ext", "nbytes"]

if REWALK or not os.path.exists(MANIFEST_CSV):
    rows = walk_patients(PATIENT_ROOT, PATIENTS, COLLECT_SIZE)
    with open(MANIFEST_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)
    print("wrote", MANIFEST_CSV)
else:
    with open(MANIFEST_CSV, newline="") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["nbytes"] = int(r["nbytes"])
    print("loaded cache", MANIFEST_CSV)

print(f"{len(rows)} files across {len({r['patient'] for r in rows})} patients")

In [ ]:
ext_counts = Counter(r["ext"] for r in rows)
print("file extensions:")
for e, n in ext_counts.most_common(20):
    tot = sum(r["nbytes"] for r in rows if r["ext"] == e and r["nbytes"] > 0)
    print(f"  {e or '(none)':<12} {n:>8}   {tot/2**30:>8.2f} GiB")

nii = [r for r in rows if NII_RE.search(r["fname"])]
print(f"\n{len(nii)} NIfTI files")

empty = [r for r in nii if 0 <= r["nbytes"] < 1024]
print(f"{len(empty)} suspiciously small (<1 KiB) NIfTIs"
      + (f", e.g. {[r['fname'] for r in empty[:3]]}" if empty else ""))

depth = Counter(r["reldir"].count("/") + (1 if r["reldir"] else 0) for r in nii)
print("\ndirectory depth below the patient folder (NIfTIs only):")
for d, n in sorted(depth.items()):
    print(f"  depth {d}: {n}")

## 2. Token census -- the naming vocabulary

This is the cell that actually drives the contrast rules below. It splits every intermediate
directory name and every filename stem into alphanumeric tokens and counts them. Contrast tags,
sequence names, visit markers and derived-file suffixes all fall out here.

In [ ]:
TOKEN_RE = re.compile(r"[^a-z0-9]+")

def toks(s):
    return [t for t in TOKEN_RE.split(str(s).lower()) if t]

def mask(s):
    """Collapse digit runs: '20190505' -> '#', 'NYU0042' -> 'NYU#'. Dates and patient IDs
    otherwise swamp every count and hide the handful of tokens that carry meaning."""
    return re.sub(r"\d+", "#", str(s))


dir_parts  = [p for r in nii for p in r["reldir"].split("/") if p]
file_stems = [stem_of(r["fname"]) for r in nii]

def show(counter, title, n=40):
    print(f"--- {title} ({len(counter)} distinct) ---")
    if not counter:
        print("  (none)")
    for t, c in counter.most_common(n):
        print(f"  {t:<24} {c}")
    print()

show(Counter(t for p in dir_parts  for t in toks(mask(p))), "directory-name tokens (masked)")
show(Counter(t for s in file_stems for t in toks(mask(s))), "filename tokens (masked)")

# short bare numbers survive masking as a category of their own -- echo index, series number,
# b-value. Long ones are dates/IDs and are already covered by the masked counts above.
show(Counter(t for s in file_stems for t in toks(s) if t.isdigit() and len(t) <= 4),
     "short numeric filename tokens (<=4 digits, unmasked)", n=20)

In [ ]:
# full directory names and full filenames -- tokens alone can hide the real convention
print("--- distinct reldir shapes, digit runs masked (top 25) ---")
for d, c in Counter(mask(r["reldir"]) for r in nii).most_common(25):
    print(f"  {c:>7}  {d or '(patient root)'}")

print("\n--- distinct filenames, digit runs masked (top 40) ---")
for f_, c in Counter(mask(r["fname"]) for r in nii).most_common(40):
    print(f"  {c:>7}  {f_}")

## 3. Classify files -> contrast labels

Ordered rules, **first match wins**, applied to `reldir + "/" + filename` lowercased. Order is
load-bearing: `seg` before everything (a `*_t1c_seg` is a label, not an image), `FLAIR` before
`T2` (`t2_flair` is FLAIR), `T1ce` before `T1`, `ADC` before `DWI`.

Every rule that matches is recorded, not just the winner, so rule collisions show up as
`ambiguous` rather than silently resolving. **Edit this table after reading section 2**, then
re-run from here -- nothing above depends on it.

In [ ]:
CONTRAST_RULES = [
    # (label, regex over the lowercased "reldir/filename")
    ("seg",   r"(seg|label|lesion|mask|tumou?r|gtv|_roi|annotation|contour)"),
    ("ADC",   r"(adc|apparent[\W_]?diff)"),
    ("DWI",   r"(dwi|dti|trace|b1000|diffusion)"),
    ("SWI",   r"(swi|gre\b|t2star|t2\*|susceptib|medic)"),
    ("FLAIR", r"(flair|dark[\W_]?fluid)"),
    # NYUMets writes the contrast-enhanced T1 as CT1 -- the modifier PRECEDES the t1, which the
    # usual "t1 + suffix" patterns all miss. Without the leading-c alternative these files fall
    # through to the T1 rule (t1 is a substring of ct1) and silently pollute the T1 bucket.
    # The guard must be a lookbehind, NOT \bct1\b: '_' is a word character, so \b never fires
    # between the '_' and the 'c' in 'NYU0001_CT1.nii.gz'. (?!\d) keeps CT10 out.
    ("T1ce",  r"((?<![a-z0-9])c[\W_]?t1(?!\d)|"
              r"t1[\W_]*(ce|c\b|post|gd|gad|contrast)|post[\W_]?contrast|"
              r"(mprage|bravo|spgr|tfe)[\W_]*(post|gd|c\b)|t1[\W_]?w?[\W_]?gd)"),
    ("T1",    r"(t1|mprage|bravo|spgr|tfe|mp[\W_]?rage)"),
    ("T2",    r"(t2|tse|cube|space)"),
]
CONTRAST_RES = [(lab, re.compile(pat, re.I)) for lab, pat in CONTRAST_RULES]
LABELS = [lab for lab, _ in CONTRAST_RULES]


def classify(reldir, fname):
    """-> (winning label or None, tuple of every label whose rule matched)."""
    s = (reldir + "/" + fname).lower()
    hits = tuple(lab for lab, rx in CONTRAST_RES if rx.search(s))
    return (hits[0] if hits else None), hits


for r in nii:
    r["contrast"], r["hits"] = classify(r["reldir"], r["fname"])

print("classified:")
for lab, n in Counter(r["contrast"] for r in nii).most_common():
    print(f"  {str(lab):<8} {n:>7}")

amb = [r for r in nii if len(r["hits"]) > 1]
print(f"\n{len(amb)} files matched >1 rule (winner = first). Collision patterns:")
for combo, n in Counter(r["hits"] for r in amb).most_common(15):
    ex = next(r["fname"] for r in amb if r["hits"] == combo)
    print(f"  {n:>7}  {' > '.join(combo):<28} e.g. {ex}")

In [ ]:
# --- the diagnostic that matters: what did NOT match any rule ---
un = [r for r in nii if r["contrast"] is None]
print(f"{len(un)} unclassified NIfTIs ({100*len(un)/max(len(nii),1):.1f}%)\n")

print("--- unclassified filenames, digit runs masked (top 40) ---")
for f_, c in Counter(mask(r["fname"]) for r in un).most_common(40):
    print(f"  {c:>7}  {f_}")

print("\n--- tokens in unclassified files, masked (top 30) ---")
for t, c in Counter(t for r in un for t in toks(mask(stem_of(r["fname"])))).most_common(30):
    print(f"  {t:<22} {c}")

print("\n--- and their parent directories, masked (top 15) ---")
for d, c in Counter(mask(r["reldir"]) for r in un).most_common(15):
    print(f"  {c:>7}  {d or '(patient root)'}")

## 4. Visit / session keys

NYUMets is longitudinal, so "which contrasts does patient X have" is really
"which contrasts does patient X have *at visit V*". We look for a session key in this order:
an ISO-ish or 8-digit date anywhere in the path, then a `ses/visit/scan/study/tp` marker, then
the first directory component below the patient folder, then `(single)` for a flat layout.

If the keys come out as dates they sort chronologically and visit ordinals are meaningful;
if not, the ordinal is just a stable arbitrary index. The cell says which case you are in.

In [ ]:
DATE_RE = re.compile(r"((?:19|20)\d{2})[-_.]?(\d{2})[-_.]?(\d{2})")
SESS_RE = re.compile(r"\b(?:ses|session|visit|scan|study|timepoint|tp)[-_ ]?(\d+)\b", re.I)


def session_key(reldir, fname):
    for src in (reldir, fname):
        m = DATE_RE.search(src)
        if m:
            return "-".join(m.groups()), "date"
    for src in (reldir, fname):
        m = SESS_RE.search(src)
        if m:
            return "ses%03d" % int(m.group(1)), "marker"
    if reldir:
        # The IMMEDIATE PARENT directory, not the first component. NYUMets nests as
        # <patient>/studyId/<STUDY_ID>/FLAIR.nii, where `studyId` is a fixed literal folder:
        # taking the first component returns "studyId" for every file, collapsing all of a
        # patient's studies into one pseudo-session whose contrasts then come from DIFFERENT
        # dates. The last component is the folder the contrasts were acquired into, which is
        # the grouping that actually means "one session" in every layout seen so far.
        return reldir.split("/")[-1], "dirname"
    return "(single)", "flat"


for r in nii:
    r["session"], r["session_src"] = session_key(r["reldir"], r["fname"])

print("session key source:", dict(Counter(r["session_src"] for r in nii)))

sess_by_pat = defaultdict(set)
for r in nii:
    sess_by_pat[r["patient"]].add(r["session"])

counts = [len(v) for v in sess_by_pat.values()]
nsess = Counter(counts)
print(f"\nsessions per patient: min {min(counts)}, max {max(counts)}, mean {np.mean(counts):.2f}")
for k in sorted(nsess)[:15]:
    print(f"  {k:>3} session(s): {nsess[k]:>5} patients")

ex = max(sess_by_pat, key=lambda p: len(sess_by_pat[p]))
print(f"\nmost visits: {ex} -> {sorted(sess_by_pat[ex])[:12]}")

## 5. Availability

In [ ]:
# (patient, session) -> {contrast: [files]}
cell = defaultdict(lambda: defaultdict(list))
for r in nii:
    if r["contrast"]:
        cell[(r["patient"], r["session"])][r["contrast"]].append(r)

pat_contrasts = defaultdict(set)          # pooled over all visits
for (p, s), d in cell.items():
    pat_contrasts[p] |= set(d)

n_pat  = len(PATIENTS)
n_sess = len(cell)
print(f"{n_pat} patients, {n_sess} (patient, session) cells\n")

print(f"{'contrast':<8} {'patients':>10} {'%':>6}   {'sessions':>10} {'%':>6}")
for lab in LABELS:
    np_ = sum(lab in v for v in pat_contrasts.values())
    ns_ = sum(lab in d for d in cell.values())
    print(f"{lab:<8} {np_:>10} {100*np_/max(n_pat,1):>5.1f}%   "
          f"{ns_:>10} {100*ns_/max(n_sess,1):>5.1f}%")

dupes = [(k, lab, len(v)) for k, d in cell.items() for lab, v in d.items() if len(v) > 1]
print(f"\n{len(dupes)} (patient, session, contrast) cells hold >1 file "
      "(repeat acquisitions, echoes, or an over-broad rule)")
for k, lab, n in dupes[:8]:
    print(f"  {k} {lab} x{n}: {[r['fname'] for r in cell[k][lab]][:3]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
xs = np.arange(len(LABELS))

axes[0].bar(xs, [sum(l in v for v in pat_contrasts.values()) for l in LABELS], color="#4878a8")
axes[0].set_title("patients with >=1 volume of each contrast")
axes[1].bar(xs, [sum(l in d for d in cell.values()) for l in LABELS], color="#a85448")
axes[1].set_title("(patient, session) cells with each contrast")
for ax in axes:
    ax.set_xticks(xs); ax.set_xticklabels(LABELS, rotation=45, ha="right")
    ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# binary availability heatmap, patients grouped by which set they have
order = sorted(pat_contrasts, key=lambda p: (tuple(l in pat_contrasts[p] for l in LABELS), p),
               reverse=True)
A = np.array([[l in pat_contrasts[p] for l in LABELS] for p in order], dtype=float)

fig, ax = plt.subplots(figsize=(0.55 * len(LABELS) + 3, 7))
ax.imshow(A, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_ylabel(f"patients (n={len(order)}, sorted by contrast set)")
ax.set_title("contrast availability per patient (pooled over visits)")
plt.tight_layout(); plt.show()

In [ ]:
# which contrast SETS occur, at the session level
combos = Counter(frozenset(d) for d in cell.values())
print(f"{len(combos)} distinct contrast sets over {n_sess} sessions\n")
print(f"{'n':>7}  {'%':>6}  set")
for cset, n in combos.most_common(20):
    print(f"{n:>7}  {100*n/n_sess:>5.1f}%  {sorted(cset, key=LABELS.index)}")

In [ ]:
# co-occurrence at the session level: P(col | row)
M = np.zeros((len(LABELS), len(LABELS)))
for d in cell.values():
    present = [i for i, l in enumerate(LABELS) if l in d]
    for i in present:
        for j in present:
            M[i, j] += 1
P = M / np.maximum(np.diag(M), 1)[:, None]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(P, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, f"{P[i,j]:.2f}", ha="center", va="center",
                color="w" if P[i, j] < .6 else "k", fontsize=8)
ax.set_title("P(column present | row present), per session")
fig.colorbar(im, ax=ax, fraction=.046); plt.tight_layout(); plt.show()

### The query that actually gates the synthesis work

How much of this dataset is usable as BraTS-style multi-contrast input, i.e. how many
**visits** carry the full quartet at once (and how many patients have at least one such visit).
Change `CORE` to test other requirements -- e.g. `("T1", "T1ce")` for a bare T1 -> T1ce bridge.

In [ ]:
CORE = ("FLAIR", "T1", "T1ce", "T2")

full_sess = [k for k, d in cell.items() if all(l in d for l in CORE)]
full_pats = {p for p, _ in full_sess}
print(f"CORE = {CORE}")
print(f"  {len(full_sess)}/{n_sess} sessions ({100*len(full_sess)/max(n_sess,1):.1f}%) complete")
print(f"  {len(full_pats)}/{n_pat} patients ({100*len(full_pats)/max(n_pat,1):.1f}%) "
      "have >=1 complete visit")

per_pat = Counter(p for p, _ in full_sess)
print(f"\ncomplete visits per patient (of those that have any): "
      f"max {max(per_pat.values()) if per_pat else 0}")
for k, v in sorted(Counter(per_pat.values()).items())[:12]:
    print(f"  {k:>3} complete visit(s): {v:>5} patients")

print("\nmissing-contrast breakdown over incomplete sessions:")
inc = [d for k, d in cell.items() if not all(l in d for l in CORE)]
for lab, c in Counter(l for d in inc for l in CORE if l not in d).most_common():
    print(f"  missing {lab:<6} {c:>7}  ({100*c/max(len(inc),1):.1f}% of incomplete)")

## 6. Header-only scan: geometry

`nib.load` is lazy -- the header comes back without touching the voxel data, so this is cheap
even over gpfs. Two things we need to know before any of this is trainable:

* is every volume on a **common grid** (already resampled / skull-stripped like BraTS), or raw
  scanner geometry with per-sequence spacing?
* within one visit, do the contrasts share an affine, i.e. are they **co-registered**?

In [ ]:
import nibabel as nib

HDR_SAMPLE = 150        # files per contrast

def header_info(path):
    img = nib.load(path)                     # lazy: no voxel data read
    hdr = img.header
    return {"shape": tuple(int(s) for s in img.shape[:3]),
            "zooms": tuple(round(float(z), 2) for z in hdr.get_zooms()[:3]),
            "axcodes": "".join(nib.aff2axcodes(img.affine)),
            "dtype": str(hdr.get_data_dtype())}


def full_path(r):
    return os.path.join(PATIENT_ROOT, r["patient"], r["reldir"], r["fname"])


rng = np.random.default_rng(0)
for lab in LABELS:
    pool = [r for r in nii if r["contrast"] == lab]
    if not pool:
        continue
    sel = [pool[i] for i in rng.permutation(len(pool))[:HDR_SAMPLE]]
    infos, bad = [], 0
    for r in sel:
        try:
            infos.append(header_info(full_path(r)))
        except Exception:
            bad += 1
    if not infos:
        print(f"{lab}: {bad} read failures\n"); continue
    print(f"--- {lab}  (n={len(infos)} sampled of {len(pool)}"
          + (f", {bad} unreadable" if bad else "") + ") ---")
    for key in ("shape", "zooms", "axcodes", "dtype"):
        top = Counter(i[key] for i in infos).most_common(4)
        print(f"  {key:<8} " + "  |  ".join(f"{v} x{c}" for v, c in top))
    print()

In [ ]:
# co-registration check: within a session, do contrasts share shape + affine?
SESS_SAMPLE = 60

keys = [k for k, d in cell.items() if len(d) >= 2]
sel = [keys[i] for i in rng.permutation(len(keys))[:SESS_SAMPLE]]

same_shape = same_affine = total = 0
for k in sel:
    firsts = [v[0] for v in cell[k].values()]
    try:
        imgs = [nib.load(full_path(r)) for r in firsts]
    except Exception:
        continue
    total += 1
    shapes = {tuple(int(s) for s in im.shape[:3]) for im in imgs}
    same_shape += len(shapes) == 1
    a0 = imgs[0].affine
    same_affine += all(np.allclose(im.affine, a0, atol=1e-3) for im in imgs)

print(f"sampled {total} multi-contrast sessions")
print(f"  identical shape  across contrasts: {same_shape}/{total}")
print(f"  identical affine across contrasts: {same_affine}/{total}")
print("\nidentical affine => already co-registered onto a common grid (BraTS-like).")
print("shapes agree but affines do not => same matrix size, different scanner geometry.")
print("neither => registration + resampling is a required preprocessing step.")

## 7. Intensity distributions per contrast

Whether the volumes share an intensity scale decides how much normalization the pipeline owes
this data. `subplot_hists` handles the four things that make these comparable -- foreground
masking, a percentile range instead of the full tail, one set of bin edges per panel, and density
rather than count -- so this cell only has to choose the voxels.

In [ ]:
from visualization import subplot_hists, subplot_images

N_SUBJECTS     = 6           # sessions to overlay
HIST_CONTRASTS = [l for l in LABELS if l != "seg"]
MAX_VOXELS     = 200_000     # cap per volume, so 6 x 7 volumes stay in memory
BG_FRAC        = 0.05        # drop voxels below BG_FRAC * p99.5 -- air and the noise floor
BINS           = 120

_rng_h = np.random.default_rng(0)


def fg_samples(path, max_voxels=MAX_VOXELS, bg_frac=BG_FRAC):
    """Foreground voxel samples from one volume.

    NYUMets is not necessarily skull-stripped, so air is a low-but-nonzero noise floor rather
    than exact zeros -- `mask=` has nothing to key on and a plain `> 0` keeps all of it.
    Thresholding relative to the volume's own robust max drops it without a brain extractor.
    Set bg_frac=0 if the data turns out to be stripped after all.
    """
    v = np.asanyarray(nib.load(path).dataobj)
    while v.ndim > 3:                      # 4D (echoes / b-values) -> first volume
        v = v[..., 0]
    v = v.astype(np.float32).ravel()
    v = v[np.isfinite(v)]
    if v.size == 0:
        return v
    v = v[v > bg_frac * np.percentile(v, 99.5)]
    if v.size > max_voxels:
        v = v[_rng_h.choice(v.size, max_voxels, replace=False)]
    return v


# sessions with the most contrasts, so the same subjects appear in as many panels as possible
pool = full_sess if full_sess else sorted(cell, key=lambda k: -len(cell[k]))
keys = pool[:N_SUBJECTS]

samples = {lab: {} for lab in HIST_CONTRASTS}
for k in keys:
    for lab in HIST_CONTRASTS:
        if lab not in cell[k]:
            continue
        try:
            s = fg_samples(full_path(cell[k][lab][0]))
        except Exception as err:
            print(f"  !! {k} {lab}: {err}")
            continue
        if s.size:
            samples[lab][k] = s

present = [l for l in HIST_CONTRASTS if samples[l]]
print(f"{len(keys)} sessions, contrasts with data: {present}")

In [ ]:
# Top row raw, bottom row divided by each volume's own p99.5. Panels are NOT share_bins --
# contrasts live on different scales and should not be forced onto one axis; within a panel
# plot_hist already pools the series to one set of edges, which is the comparison that matters.
N = len(present)
lab_of = lambda k: f"{k[0]}/{k[1]}"

panels  = [{lab_of(k): v for k, v in samples[l].items()} for l in present]
panels += [{lab_of(k): v / np.percentile(v, 99.5) for k, v in samples[l].items()}
           for l in present]

# plot_hist passes color=None straight to ax.hist, and an EXPLICIT None disables matplotlib's
# property cycle -- every step-histogram then draws black and the legend is useless. So name the
# colours. As a TUPLE, not a list: subplot_hists spreads any *list* whose length matches the
# panel count across the panels, which would silently fire when len(keys) == 2*len(present).
_cyc = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors = tuple(_cyc[i % len(_cyc)] for i in range(len(keys)))

subplot_hists(
    panels,
    ncols=N,
    titles=[f"{l}  (n={len(samples[l])})" for l in present] + [f"{l} / p99.5" for l in present],
    bins=BINS,
    p=(0.5, 99.5),                                   # raw row: percentile range per panel
    range=[None] * N + [(0.0, 1.4)] * N,             # normalised row: one fixed range
    color=colors,
    xlabel=["raw intensity"] * N + ["/ per-volume p99.5"] * N,
    legend=[True] + [False] * (2 * N - 1),           # one legend is enough for the figure
    panel_size=(3.3, 3.0),
    suptitle="voxel intensity distribution per contrast, one line per session",
)

In [ ]:
# --- the number that decides whether per-subject normalisation is mandatory ---
print(f"{'contrast':<8} {'n':>3} {'med p50':>10} {'med p95':>10} {'med p99.5':>11} "
      f"{'p99.5 max/min':>14}")
for lab in present:
    q = np.array([np.percentile(v, [50, 95, 99.5]) for v in samples[lab].values()])
    spread = q[:, 2].max() / max(q[:, 2].min(), 1e-9)
    print(f"{lab:<8} {len(q):>3} {np.median(q[:,0]):>10.1f} {np.median(q[:,1]):>10.1f} "
          f"{np.median(q[:,2]):>11.1f} {spread:>14.2f}")

print("\np99.5 max/min ~1 => volumes already share a scale; >>1 => per-subject normalisation")
print("is required before any additive/percentile contrast model. If the bottom-row curves")
print("superimpose, dividing by p99.5 is sufficient; if they do not, the tail is being set by")
print("pathology rather than normal tissue and a lower percentile is needed.")
print("\nADC is a quantitative map in physical units, so its spread should be ~1.0 even when")
print("the weighted contrasts scatter. If it is not, suspect the ADC files, not the scaling.")

## 8. Eyeball one complete visit

In [ ]:
def mid_slice(path):
    """Middle axial slice of a volume, canonicalised to RAS so orientation is comparable."""
    img = nib.as_closest_canonical(nib.load(path))
    vol = np.asanyarray(img.dataobj)
    while vol.ndim > 3:
        vol = vol[..., 0]
    z = vol.shape[2] // 2
    return np.rot90(vol[:, :, z].astype(np.float32))


key = full_sess[0] if full_sess else max(cell, key=lambda k: len(cell[k]))
d = cell[key]
labs = sorted(d, key=LABELS.index)
print("showing", key, "->", labs)

imgs = [mid_slice(full_path(d[lab][0])) for lab in labs]

# share_window=False is the point: each contrast gets its own percentile stretch. The default
# shared window would scale every panel to whichever contrast has the largest dynamic range and
# black out the rest -- exactly what section 7's raw row shows happening.
subplot_images(
    imgs,
    titles=[f"{lab}  {im.shape}" for lab, im in zip(labs, imgs)],
    p=(1, 99),
    share_window=False,
    suptitle=f"{key[0]}   {key[1]}",
)

## 9. Tidy CSV

One row per `(patient, session, contrast, file)` with the path relative to `PATIENT_ROOT`.
This is what a `datasets/NYUMets/` registrar should read later, so it does not have to
re-derive any of the above.

In [ ]:
sess_order = {}
for p, ss in sess_by_pat.items():
    for i, s in enumerate(sorted(ss)):
        sess_order[(p, s)] = i

out = []
for (p, s), d in sorted(cell.items()):
    for lab in sorted(d, key=LABELS.index):
        for r in d[lab]:
            out.append({"patient": p, "session": s, "visit_idx": sess_order[(p, s)],
                        "contrast": lab, "n_at_cell": len(d[lab]),
                        "relpath": "/".join(x for x in (p, r["reldir"], r["fname"]) if x),
                        "nbytes": r["nbytes"]})

cols = ["patient", "session", "visit_idx", "contrast", "n_at_cell", "relpath", "nbytes"]
with open(TIDY_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(out)

print(f"wrote {len(out)} rows -> {TIDY_CSV}")
print("root for relpath:", PATIENT_ROOT)
for r in out[:5]:
    print(" ", r)

## Are a patient's studies aligned? (visual)

Point `ROOT` at the unregistered h5s or at a registered copy -- nothing else changes, so the
same two cells serve as the before and the after.

`Z` is an ORIGINAL slice index (what `slice_index` stores), so it names the same level in both
sets. `DZ`, `DH`, `DW` are applied to the **other** study only: dial them until the two line up,
and what you land on IS the offset registration should have found.

Read the last two columns. **difference**: structure cancels when aligned, leaving noise and
genuine interval change; a bright edge-shaped ring means a shift. **checkerboard**: edges run
straight across the tile boundaries when aligned, and step at every boundary when not.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from scripts.register_nyumets import index_patients
from preprocessing.nyumets_h5 import CONTRASTS, translate
from visualization.image import set_display_orient, subplot_images

set_display_orient("radiological")

ROOT = "../datasets/NYUMets_h5"     # ..._reg for the registered copy
SOURCE = "raw"                      # "raw" = img_raw, the WHOLE HEAD. "norm" = img_median_mad,
                                    # which cmap.normalize_masked zeroes outside each study's
                                    # own brain mask -- so two studies masked differently look
                                    # misaligned even when they are not. Judge alignment on raw.
PATIENT = None                      # None = the first patient with >= 2 studies
CUR, OTH = 0, 1                     # which two studies of that patient (sorted by session)
Z = None                            # original slice index; None = middle of the shared range
DZ, DH, DW = 0, 0, 0                # offset applied to the OTHER study's slice
P = (1, 99)                         # display percentiles

# The notebook's cwd is usually notebooks/, so a path written relative to the repo root has to
# be resolved against REPO rather than against the cwd.
import os as _os
_cands = [ROOT, _os.path.join(globals().get("REPO", "."), ROOT),
          _os.path.expanduser("~/scratch/datasets/" + _os.path.basename(ROOT))]
ROOT = next((_os.path.abspath(c) for c in _cands if _os.path.isdir(c)), ROOT)
print("root:", ROOT)

T1, CT1 = CONTRASTS.index("T1"), CONTRASTS.index("T1ce")
_pats = index_patients(ROOT)
_multi = {k: v for k, v in _pats.items() if len(v) > 1}
if not _multi:
    raise RuntimeError(f"no patient under {ROOT} has two studies")
_pid = PATIENT or sorted(_multi)[0]
_st = sorted(_multi[_pid], key=lambda s: s.case)
cur, oth = _st[CUR], _st[OTH]

_shared = sorted(set(cur.index.tolist()) & set(oth.index.tolist()))
_z = int(_shared[len(_shared) // 2]) if Z is None else int(Z)
print(f"patient {_pid}: {len(_st)} studies -> {[s.case for s in _st]}")
for s in (cur, oth):
    reg = (f"  reg_applied={bool(s.attrs.get('reg_applied'))} "
           f"offset={tuple(np.asarray(s.attrs['reg_offset']).tolist())}"
           if "reg_offset" in s.attrs else "  (no reg attrs: built before registration)")
    print(f"  {s.case:<28s} depth {s.depth:>4d}  kept {s.index.min():>3d}..{s.index.max():<3d}"
          + reg)
print(f"z={_z} of {cur.case}  vs  z={_z + DZ} of {oth.case}   "
      f"(DZ={DZ}, DH={DH}, DW={DW}); {len(_shared)} common slice indices")


def plane(st, contrast, z, dh=0, dw=0):
    """(H, W) of one contrast at ORIGINAL slice index z; blank if that slice was not kept."""
    v = st.planes(contrast, SOURCE)               # (D, H, W) on the original z axis
    if not (0 <= z < v.shape[0]) or not st.acquired[z]:
        return torch.zeros(v.shape[1:]), False
    p = v[z]
    return (translate(p, (dh, dw)) if (dh or dw) else p), True


_a_t1, _ok1 = plane(cur, T1, _z)
_a_ct, _ = plane(cur, CT1, _z)
_b_t1, _ok2 = plane(oth, T1, _z + DZ, DH, DW)
_b_ct, _ = plane(oth, CT1, _z + DZ, DH, DW)
if not _ok1:
    print(f"  !! {cur.case} has no slice at z={_z}")
if not _ok2:
    print(f"  !! {oth.case} has no slice at z={_z + DZ} -- change Z or DZ")

_v = float(torch.quantile(torch.stack([_a_t1, _b_t1]).abs().float().flatten(), 0.99)) or 1.0
_H, _W = _a_t1.shape
_k = max(8, _W // 8)
_yy, _xx = np.mgrid[0:_H, 0:_W]
_m = torch.from_numpy((((_yy // _k) + (_xx // _k)) % 2).astype(bool))
_rows = [[_a_t1, _b_t1, _a_t1 - _b_t1, torch.where(_m, _a_t1, _b_t1)],
         [_a_ct, _b_ct, _a_ct - _b_ct, torch.where(_m, _a_ct, _b_ct)]]
fig, _ = subplot_images(
    _rows, row_labels=["T1", "CT1"],
    col_titles=[f"{cur.case}\nz={_z}", f"{oth.case}\nz={_z + DZ}", "difference", "checkerboard"],
    cmap=["gray", "gray", "RdBu_r", "gray"],
    vmin=[None, None, -_v, None], vmax=[None, None, _v, None],
    p=P, magnitude=False, panel_size=(3.0, 3.2),
    suptitle=f"{_pid}   {SOURCE}   DZ={DZ}, DH={DH}, DW={DW}", show=False)
plt.show()

### Which `DZ` lines up?

The other study's T1 across a range of `DZ`, with the current study's T1 on the left to compare
against. `score` is the correlation inside the two brain masks -- higher is better, and the peak
should sit at the same `DZ` your eye picks. A flat score with no clear peak means no
through-plane shift explains the pair: different slice spacing, or rotation.

In [ ]:
DZ_RANGE = range(-8, 9, 2)          # which offsets to try

_ref, _ = plane(cur, T1, _z)
_ref_m = cur.masks()[_z].bool() if "masks" in dir(cur) else (_ref.abs() > 0)
_panels, _titles, _scores = [_ref], [f"{cur.case}\nz={_z} (reference)"], []
for _d in DZ_RANGE:
    _p, _ok = plane(oth, T1, _z + _d, DH, DW)
    _om = oth.masks()[_z + _d].bool() if (0 <= _z + _d < oth.depth) else torch.zeros_like(_ref_m)
    if (DH or DW):
        _om = translate(_om, (DH, DW))
    _joint = _ref_m & _om
    if _ok and int(_joint.sum()) > 50:
        _x, _y = _ref[_joint].float(), _p[_joint].float()
        _x, _y = _x - _x.mean(), _y - _y.mean()
        _s = float((_x * _y).sum() / (_x.norm() * _y.norm()).clamp(min=1e-8))
    else:
        _s = float("nan")
    _scores.append((_d, _s))
    _panels.append(_p)
    _titles.append(f"DZ={_d:+d}\nscore {_s:.3f}")

print("  DZ   score (masked correlation, higher is better)")
for _d, _s in _scores:
    # a DZ with too little mask overlap scores NaN; show it as such rather than crashing
    _bar = "" if _s != _s else "#" * int(max(_s, 0.0) * 40)
    print(f"  {_d:+3d}   {_s:6.3f}  {_bar}")
_best = max((s for s in _scores if s[1] == s[1]), key=lambda t: t[1], default=None)
print(f"\n  best DZ = {_best[0]:+d}  (score {_best[1]:.3f})" if _best else "\n  no usable overlap")

fig, _ = subplot_images(
    [_panels], col_titles=_titles, cmap="gray", p=P, magnitude=False,
    panel_size=(2.3, 2.6), suptitle=f"{_pid}: {oth.case} across DZ (DH={DH}, DW={DW})",
    show=False)
plt.show()

### Where should that offset have come from?

You dialled in a `DZ` by eye. This checks it against the two sources that claim to know it: the
stored **affines** (exact, if the studies differ by a translation) and **cross-correlation** on
the pixels. It also splits the prediction into the part that comes from the scanner's world
origins and the part the builder's own centre crop/pad introduced -- `--crop 224` aligns FOV
centres by index, so two studies with different `native_size` are offset in-plane before anyone
touches them. There is no crop in z, so `dz` is pure origin difference.

If `affine says` matches what your eye found, the offset is analytically known and correlation
is not needed. If they disagree, believe your eye and say so -- the affines may not be
trustworthy for this cohort.

In [ ]:
from preprocessing.nyumets_h5 import affine_offset, lowpass_inplane, translation_offset

_crop = max(0, int(cur.attrs.get("crop_size", 0) or 0))   # write_h5 stores -1 for "no crop"
_key = lambda st: (np.asarray(st.attrs.get("affine", np.eye(4)), dtype=np.float64),
                   tuple(int(v) for v in st.attrs.get("native_size", (_crop, _crop))),
                   st.depth)
_t_aff, _ob = affine_offset(_key(cur), _key(oth), _crop)

# cross-correlation on the SAME source the figures use, whole volume
_va = cur.planes(T1, SOURCE)
_vb = oth.planes(T1, SOURCE)
_d = max(_va.shape[0], _vb.shape[0])
_pad = lambda v: (v if v.shape[0] >= _d else
                  torch.cat([v, torch.zeros((_d - v.shape[0],) + v.shape[1:])]))
_t_xc = translation_offset(lowpass_inplane(_pad(_va), 0.25), lowpass_inplane(_pad(_vb), 0.25))

print(f"{cur.case}  vs  {oth.case}      (source={SOURCE}, crop={_crop})")
print(f"  you dialled in   dz={DZ:+4d}  dh={DH:+4d}  dw={DW:+4d}")
print(f"  affine says      dz={_t_aff[0]:+4d}  dh={_t_aff[1]:+4d}  dw={_t_aff[2]:+4d}"
      f"      obliqueness {_ob:.4f}")
print(f"  cross-corr says  dz={_t_xc[0]:+4d}  dh={_t_xc[1]:+4d}  dw={_t_xc[2]:+4d}")

# Where the affine prediction comes from: world origins vs the builder's centre crop.
_oz = float(np.asarray(cur.attrs["affine"])[2, 3] - np.asarray(oth.attrs["affine"])[2, 3]) \
    if "affine" in cur.attrs and "affine" in oth.attrs else float("nan")
_nc = tuple(int(v) for v in cur.attrs.get("native_size", (_crop, _crop)))
_no = tuple(int(v) for v in oth.attrs.get("native_size", (_crop, _crop)))
_crop_dh = ((_crop - _no[0]) // 2) - ((_crop - _nc[0]) // 2) if _crop else 0
_crop_dw = ((_crop - _no[1]) // 2) - ((_crop - _nc[1]) // 2) if _crop else 0
print(f"\n  breakdown")
print(f"    z world origins differ by {_oz:+.1f} mm  ->  dz is pure origin (no crop in z)")
print(f"    native {_nc} vs {_no}  ->  centre crop/pad alone contributes "
      f"dh={_crop_dh:+d}, dw={_crop_dw:+d}")
print(f"    depths {cur.depth} vs {oth.depth}, kept {cur.index.min()}..{cur.index.max()} vs "
      f"{oth.index.min()}..{oth.index.max()}")

_agree = (abs(_t_aff[0] - DZ) <= 1, abs(_t_xc[0] - DZ) <= 1)
print(f"\n  affine matches your DZ: {_agree[0]}        cross-corr matches your DZ: {_agree[1]}")
if _ob > 0.02:
    print(f"  !! obliqueness {_ob:.4f}: these two are ROTATED relative to each other, and no "
          f"integer shift aligns them.")

## Did registration work? All studies of a patient on one grid, through the loader

Reads the registered set through `NYUMetsGuidedDataset` (the same `_read` / `_other_study_slice`
path training uses) and checks three things for every multi-study patient:

1. **bookkeeping**: every study carries `reg_applied`, the same `reg_reference`, the same `affine`
   (the reference's grid) and an IDENTICAL `slice_index`. Registration writes exactly that, so a
   mismatch means the study was skipped, or the file came from a build before registration.
2. **loader pairing**: for each kept slice, the slice the loader hands back as the other-study
   guide (`guide_slice="index"` and `"matched"`) has the same ORIGINAL slice index.
3. **pixels**: correlation of `CONTRAST` between each study and the patient's first study, inside
   the joint brain mask, probed at small shifts along each axis. Registered => the peak sits at
   (0, 0, 0). A peak elsewhere with a real `gain` is the residual offset, in voxels.

The figure shows the lowest-scoring patient (or `SHOW`) at one slice: studies on top, the
difference from the reference in the middle (structure should cancel), and a checkerboard at the
bottom (edges should run straight across tile boundaries).

In [ ]:
from types import SimpleNamespace

import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets.NYUMets.longitudinal_dataset import NYUMetsGuidedDataset
from preprocessing.nyumets_h5 import CONTRASTS
from visualization.image import subplot_images

REG_ROOT   = "../datasets/NYUMets_h5_reg"   # the registered copy (or a split of it)
IMAGE_KEY  = "img_median_mad"               # what the loader trains on
SCALES     = [3.0, 3.0, 3.0, 3.0]
CONTRAST   = "T1"
N_PATIENTS = 40                             # None = every multi-study patient
Z_STEP     = 2                              # score every Z_STEP-th slice (speed only)
SHIFTS     = range(-3, 4)                   # probe offsets per axis, voxels
SHOW       = None                           # patient to plot; None = lowest ncc@0
VMIN, VMAX = -1.0, 1.0

import os as _os
_cands = [REG_ROOT, _os.path.join(globals().get("REPO", "."), REG_ROOT),
          _os.path.expanduser("~/scratch/datasets/" + _os.path.basename(REG_ROOT))]
REG_ROOT = next((_os.path.abspath(c) for c in _cands if _os.path.isdir(c)), REG_ROOT)
print("root:", REG_ROOT)

# other_study drops single-study patients for us; everything else at the dataset defaults
ds = NYUMetsGuidedDataset(SimpleNamespace(
    root=REG_ROOT, image_key=IMAGE_KEY, scales=SCALES,
    guide_mode="other_study", guide_slice="index", deterministic=True))
CI = CONTRASTS.index(CONTRAST)

by_pat = {}
for fi, p in enumerate(ds.patients):
    by_pat.setdefault(p, []).append(fi)
pats = sorted(by_pat)
if N_PATIENTS:
    pats = [pats[i] for i in sorted(np.random.default_rng(0).permutation(len(pats))[:N_PATIENTS])]
print(f"{len(by_pat)} multi-study patients, checking {len(pats)}")


def volume(fi):
    """(n, H, W) of CONTRAST and the brain mask, slice by slice through the loader's own _read."""
    n = ds.n_slices[fi]
    v = np.stack([ds._read(fi, li)[0][..., CI] for li in range(n)]).astype(np.float32)
    m = np.asarray(ds._handle(ds.img_paths[fi])["mask"][:, :, :, 0]).astype(bool)
    return v, m


def _ov(n, d):
    """a[z] vs b[z + d] over the overlap -> (slice into a, slice into b)."""
    return slice(max(0, -d), n - max(0, d)), slice(max(0, d), n - max(0, -d))


def ncc(a, b, ma, mb, t):
    """Correlation of a vs b shifted by t = (dz, dh, dw), inside both brain masks."""
    sa, sb = zip(*(_ov(n, d) for n, d in zip(a.shape, t)))
    sa = (slice(sa[0].start, sa[0].stop, Z_STEP),) + sa[1:]
    sb = (slice(sb[0].start, sb[0].stop, Z_STEP),) + sb[1:]
    j = ma[sa] & mb[sb]
    if j.sum() < 1000:
        return float("nan")
    x, y = a[sa][j], b[sb][j]
    x, y = x - x.mean(), y - y.mean()
    return float((x * y).sum() / max(np.sqrt((x * x).sum() * (y * y).sum()), 1e-8))


rows, cache = [], {}
for p in pats:
    fis = sorted(by_pat[p], key=lambda f: ds.img_paths[f])
    attrs = [dict(ds._handle(ds.img_paths[fi]).attrs) for fi in fis]
    zidx = [ds._slice_index(fi) for fi in fis]
    books = {
        "reg_applied": all(bool(a.get("reg_applied", False)) for a in attrs),
        "one_reference": len({str(a.get("reg_reference", "?")) for a in attrs}) == 1,
        "same_affine": all(np.allclose(np.asarray(a.get("affine", np.eye(4))),
                                       np.asarray(attrs[0].get("affine", np.eye(4))), atol=1e-3)
                           for a in attrs),
        "same_slice_index": all(np.array_equal(z, zidx[0]) for z in zidx),
    }

    # what the loader actually pairs: original index of the guide slice == that of the target
    loader_ok = True
    for mode in ("index", "matched"):
        ds.guide_slice = mode
        for fi in fis:
            zt = ds._slice_index(fi)
            for li in range(0, ds.n_slices[fi], Z_STEP):
                gf, gz = ds._other_study_slice(fi, li, li, 0)
                loader_ok &= int(ds._slice_index(gf)[gz]) == int(zt[li])
    ds.guide_slice = "index"

    vols = [volume(fi) for fi in fis]
    cache[p] = (fis, vols)
    a, ma = vols[0]
    for k in range(1, len(fis)):
        b, mb = vols[k]
        if a.shape != b.shape:
            rows.append(dict(patient=p, pair=k, **books, loader_ok=loader_ok,
                             ncc0=float("nan"), best=None, gain=float("nan"),
                             note=f"shape {a.shape} vs {b.shape}"))
            continue
        n0 = ncc(a, b, ma, mb, (0, 0, 0))
        best, top = [], n0
        for ax in range(3):
            s = {d: ncc(a, b, ma, mb, tuple(d if i == ax else 0 for i in range(3)))
                 for d in SHIFTS}
            d_best = max((d for d in s if s[d] == s[d]), key=s.get, default=0)
            best.append(d_best)
            top = max(top, s[d_best])
        rows.append(dict(patient=p, pair=k, **books, loader_ok=loader_ok,
                         ncc0=n0, best=tuple(best), gain=top - n0, note=""))
    if len(cache) > 3:                      # keep only a few patients' volumes for the figure
        cache.pop(next(iter(cache)))

# ---------------- report ----------------
BOOK = ("reg_applied", "one_reference", "same_affine", "same_slice_index", "loader_ok")
print(f"\n{'patient':<14} {'pair':>4} " + " ".join(f"{c[:10]:>10}" for c in BOOK)
      + f" {'ncc@0':>7} {'peak dz,dh,dw':>14} {'gain':>6}")
for r in rows:
    flag = "" if (all(r[c] for c in BOOK) and r["best"] == (0, 0, 0)) else "  <--"
    print(f"{r['patient']:<14} {r['pair']:>4} " + " ".join(f"{str(r[c]):>10}" for c in BOOK)
          + f" {r['ncc0']:7.3f} {str(r['best']):>14} {r['gain']:6.3f} {r['note']}{flag}")

n = len(rows)
peak0 = sum(r["best"] == (0, 0, 0) for r in rows)
near0 = sum(r["best"] is not None and max(map(abs, r["best"])) <= 1 for r in rows)
print(f"\n{n} study pairs over {len(pats)} patients")
for c in BOOK:
    print(f"  {c:<18} {sum(bool(r[c]) for r in rows):>4}/{n}")
print(f"  peak at (0,0,0)    {peak0:>4}/{n}     within 1 voxel: {near0}/{n}")
print(f"  median ncc@0 {np.nanmedian([r['ncc0'] for r in rows]):.3f}   "
      f"worst {np.nanmin([r['ncc0'] for r in rows]):.3f}")
print("\nA peak off 0 with a gain of a few thousandths is noise between visits; a gain of"
      "\n>~0.02 at |shift| >= 2 is a real residual offset for that pair.")

# ---------------- figure: one patient, one slice, all studies ----------------
show = SHOW or min(rows, key=lambda r: r["ncc0"] if r["ncc0"] == r["ncc0"] else -1)["patient"]
if show not in cache:
    fis = sorted(by_pat[show], key=lambda f: ds.img_paths[f])
    cache[show] = (fis, [volume(fi) for fi in fis])
fis, vols = cache[show]
li = int(np.argmax(vols[0][1].sum((1, 2))))          # the slice with the most brain
ref = vols[0][0][li]
H, W = ref.shape
k = max(8, W // 8)
yy, xx = np.mgrid[0:H, 0:W]
chk = ((yy // k) + (xx // k)) % 2 == 1
imgs = [v[li] if li < v.shape[0] else np.zeros_like(ref) for v, _ in vols]
names = [_os.path.basename(_os.path.dirname(ds.img_paths[fi])) for fi in fis]
z0 = int(ds._slice_index(fis[0])[li])

C = len(imgs)
grid = [imgs,
        [None] + [im - ref for im in imgs[1:]],
        [None] + [np.where(chk, ref, im) for im in imgs[1:]]]
subplot_images(
    [[None if x is None else torch.from_numpy(np.ascontiguousarray(x)) for x in r] for r in grid],
    row_labels=[CONTRAST, "", ""],            # col 0 is blank below row 0, so label via xlabels
    xlabels=[[None] * C, [None] + ["minus ref"] * (C - 1), [None] + ["checker vs ref"] * (C - 1)],
    col_titles=[nm + (" (ref)" if i == 0 else "") for i, nm in enumerate(names)],
    cmap=[["gray"] * C, ["RdBu_r"] * C, ["gray"] * C],
    vmin=[[VMIN] * C, [-VMAX] * C, [VMIN] * C],
    vmax=[[VMAX] * C, [VMAX] * C, [VMAX] * C],
    magnitude=False, panel_size=(3.0, 3.2),
    suptitle=f"{show}: local slice {li} (original z={z0}), {IMAGE_KEY} / {SCALES[CI]:g}")